In [1]:
import json
import tiktoken
import re
from collections import defaultdict
from openai import OpenAI
from tqdm.auto import tqdm

/home/luis/miniconda3/envs/doc/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "server").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
print(f"Added {ROOT} to sys.path")

Added /home/luis/Documents/FGV/Laboratory/document-graph to sys.path


In [3]:
from server.utils.config import Config
from server.schemas.types import Node, Edge, Graph

In [4]:
infra_path = "../../infra/"
json_path = f"{infra_path}json/"
graph_json = f"{json_path}graph/"
graph_test = [
    f"{graph_json}BELLICUMPHARMACEUTICALS_INC_05_07_2019-EX-10.1-Supply_Agreement.json", # 0.817654
    f"{graph_json}EdietsComInc_20001030_10QSB_EX-10.4_2606646_EX-10.4_Co-Branding_Agreement.json", # 0.206410
    f"{graph_json}HealthcentralCom_19991108_S-1A_EX-10.27_6623292_EX-10.27_Co-Branding_Agreement.json", # 0.090494 
    f"{graph_json}RitterPharmaceuticalsInc_20200313_S-4A_EX-10.54_12055220_EX-10.54_Development_Agreement.json", # 0.347634
    f"{graph_json}TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding_Agreement.json" # 0.360902
]

graph_json, graph_test

('../../infra/json/graph/',
 ['../../infra/json/graph/BELLICUMPHARMACEUTICALS_INC_05_07_2019-EX-10.1-Supply_Agreement.json',
  '../../infra/json/graph/EdietsComInc_20001030_10QSB_EX-10.4_2606646_EX-10.4_Co-Branding_Agreement.json',
  '../../infra/json/graph/HealthcentralCom_19991108_S-1A_EX-10.27_6623292_EX-10.27_Co-Branding_Agreement.json',
  '../../infra/json/graph/RitterPharmaceuticalsInc_20200313_S-4A_EX-10.54_12055220_EX-10.54_Development_Agreement.json',
  '../../infra/json/graph/TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding_Agreement.json'])

In [5]:
with open(graph_test[4], "r") as f:
    graph = json.load(f)

nodes = [Node(**node) for node in graph["nodes"]]
edges = [Edge(**edge) for edge in graph["edges"]]

len(nodes), len(edges)

(270, 237)

#### inverted index

In [6]:
def tokenize(text: str):
    text = text.lower()
    return re.findall(r"\b\w+\b", text)

# MAP
def map_nodes(nodes):
    pairs = []

    for node in nodes:
        words = tokenize(node.text)

        for word in words:
            pairs.append((word, node.id))

    return pairs

In [7]:
def shuffle(pairs):
    groups = defaultdict(list)

    for word, node_id in pairs:
        groups[word].append(node_id)

    return groups

In [8]:
def reduce_groups(groups):
    inverted_index = {}

    for word, node_ids in groups.items():
        inverted_index[word] = list(set(node_ids))

    return inverted_index

In [9]:
mapped = map_nodes(nodes)
grouped = shuffle(mapped)
inverted_index = reduce_groups(grouped)

In [10]:
query = "payment"
result = inverted_index.get(query.lower(), [])
print(len(result), result)

12 ['root::TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement-p-128', 'root::TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement-p-175', 'root::TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement-p-114', 'root::TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement-p-209', 'root::TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement-p-158', 'root::TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement-p-185', 'root::TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement-p-162', 'root::TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement-p-161', 'root::TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement-p-131', 'root::TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement-p-173', 'root::TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement-p-168', 'root::TomOnlineInc_20060501_20-F_EX-4.

#### create KG

In [11]:
## create knowledge graph from text nodes
# ------------------------V5 PROMPT TEXT FULL------------------------
SYSTEM_PROMPT = """You will receive the full details of ONE contract paragraph node: {id, text, paragraph_enum, relationsCount}. Treat this paragraph as the source evidence for extraction. Do not invent section numbers, titles, or hierarchy not explicitly supported by the text."""

PROMPT_INSTRUCTION = """
Your task is to act as a legal contract graph extractor following a contract ontology. From this single paragraph, create a self-contained set of nodes and edges that are explicitly supported by the text.

The input unit is a paragraph node, not a guaranteed formal clause header. Therefore, preserve the paragraph as the primary structural unit unless the text itself explicitly states a clause or section reference.

Output ONLY a single, strict JSON object with this structure:

{
  "contract_id": "...",
  "source_paragraph_id": "...",
  "nodes": [ ... ],
  "edges": [ ... ]
}

Use only facts explicitly supported by the text. If something is ambiguous, omit it.

──────────────────────────
REASONING PROCESS
──────────────────────────
1. Isolate the contractual prose and ignore noise such as page headers, redaction notices, tables of contents, or formatting artifacts.
2. Create one structural `CLAUSE` node representing the source paragraph itself.
3. Identify explicit legal entities, defined terms, references, values, obligations, rights, prohibitions, and conditions stated in the paragraph.
4. Create only nodes that are textually grounded in the paragraph.
5. Connect the nodes using the edge types defined below.

──────────────────────────
NODE TYPES
──────────────────────────
1. CLAUSE
Format:
{
  "id": "clause:<source paragraph id>",
  "node_type": "CLAUSE",
  "source_paragraph_id": "<original paragraph id>",
  "text": "<full paragraph text>",
  "paragraph_enum": <int>,
  "clause_id": "<explicit section/clause label if present, else null>",
  "title": "<explicit title if present, else null>",
  "level": <int or null>
}
Rules:
- Always create exactly one primary `CLAUSE` node for the source paragraph.
- `clause_id`, `title`, and `level` are optional. Set them to null if not explicit in the text.
- Do not infer hierarchy from paragraph IDs such as `...-p-391`.

2. DEFINED_TERM
Format:
{ "id": "term:<Canonical Term>", "node_type": "DEFINED_TERM", "term": "<Canonical Term>", "definition": "<definition text or null>" }
Rules:
- Create when the paragraph explicitly defines a term or clearly uses a contract-defined capitalized term.
- Canonicalize to the clean term name.

3. PARTY
Format:
{ "id": "party:<Party Name>", "node_type": "PARTY", "name": "<Party Name>", "role": "<explicit role or null>", "address": "<explicit address or null>" }
Rules:
- Create for legal entities, individuals, and explicit contract roles such as Supplier, Buyer, Licensor, Licensee, Employer, Employee.
- Only assign `role` if it is explicit in the text or directly stated through an alias definition.

4. OBLIGATION
Format:
{ "id": "obligation:<short canonical action>", "node_type": "OBLIGATION", "action": "<required action>", "deadline": "<explicit deadline or null>" }
Rules:
- Create when the text imposes a duty, usually expressed with `shall`, `must`, `is required to`, or equivalent mandatory language.

5. RIGHT
Format:
{ "id": "right:<short canonical action>", "node_type": "RIGHT", "action": "<permitted action>", "frequency": "<explicit frequency or null>" }
Rules:
- Create when the text grants discretion, entitlement, permission, or power, usually expressed with `may`, `is entitled to`, `has the right to`.

6. PROHIBITION
Format:
{ "id": "prohibition:<short canonical action>", "node_type": "PROHIBITION", "action": "<forbidden action>" }
Rules:
- Create when the text forbids conduct, usually expressed with `shall not`, `may not`, `must not`, `is prohibited from`.

7. CONDITION
Format:
{ "id": "condition:<short canonical trigger>", "node_type": "CONDITION", "trigger": "<trigger text>", "operator": "IF|UNLESS|SUBJECT TO|WHEN|AFTER|BEFORE|OTHER" }
Rules:
- Create when a prerequisite, trigger, dependency, or activating condition is explicit.

8. REFERENCE
Format:
{ "id": "reference:<canonical reference>", "node_type": "REFERENCE", "name": "<name>", "citation": "<citation or null>" }
Rules:
- Create for external laws, standards, documents, regulatory authorities, or named non-contract references.
- Internal cross-references to sections or articles of the same contract should be modeled as `CLAUSE` nodes, not `REFERENCE` nodes.

9. VALUE
Format:
{ "id": "value:<literal text>", "node_type": "VALUE", "value_type": "Currency|Percentage|Days|Months|Years|Quantity|Date|Other", "amount": "<amount or literal>", "unit": "<unit or null>" }
Rules:
- Create for explicit quantities, amounts, percentages, deadlines, durations, and dates.

──────────────────────────
EDGE TYPES
──────────────────────────
Format: { "src": "<id>", "tgt": "<id>", "type": "<EDGE_TYPE>" }

- CONTAINS: CLAUSE -> any extracted node contained in the paragraph.
- DEFINES: CLAUSE -> DEFINED_TERM when the paragraph defines the term.
- USES: CLAUSE -> DEFINED_TERM when the term is used but not defined here.
- MENTIONS_PARTY: CLAUSE -> PARTY for every explicitly mentioned party.
- REFERENCES: CLAUSE -> CLAUSE for explicit internal cross-references such as Section 3.1 or Article 10.
- CITES: CLAUSE -> REFERENCE for explicit external sources, standards, laws, or documents.
- ASSIGNS_OBLIGATION_TO: OBLIGATION -> PARTY when the obligated actor is explicit.
- GRANTS_RIGHT_TO: RIGHT -> PARTY when the holder is explicit.
- ASSIGNS_PROHIBITION_TO: PROHIBITION -> PARTY when the restricted subject is explicit.
- DEPENDS_ON: OBLIGATION|RIGHT|PROHIBITION|CLAUSE -> CONDITION when activation is conditional.
- HAS_VALUE: OBLIGATION|RIGHT|PROHIBITION|CONDITION|CLAUSE -> VALUE when the value qualifies that node.

──────────────────────────
CRITICAL RULES
──────────────────────────
1. Extract only what is explicitly supported by the paragraph text.
2. Do not invent a parent clause, section title, or clause hierarchy unless it is expressly present.
3. If an internal section or article is cited, create a minimal `CLAUSE` node for that cited target with only the explicit citation as its `clause_id`, and connect it with `REFERENCES`.
4. Create unique nodes and unique edges only once.
5. Sort nodes by `id` and edges by `src`, then `type`, then `tgt`.
6. If the paragraph contains no extractable legal information, return empty `nodes` and `edges` arrays, except you may still include the primary `CLAUSE` node if contractual prose is present.
7. Prefer omission over speculation."""

In [12]:
def build_extraction_inputs(nodes):
    complete_graph = []
    for node in nodes:
        complete_graph.append({
            "id": node.id,
            "text": node.text,
            "paragraph_enum": node.paragraph_enum,
            "relationsCount": node.relationsCount,
        })
    return complete_graph

def format_extraction_messages(paragraph_node):
    system_message = SYSTEM_PROMPT
    user_message = (
        f"{PROMPT_INSTRUCTION}\n\n"
        "SOURCE PARAGRAPH NODE\n"
        f"{json.dumps(paragraph_node, ensure_ascii=True, indent=2)}"
    )
    return {
        "system": system_message,
        "user": user_message,
    }


paragraph_inputs = build_extraction_inputs(nodes)
paragraph_inputs[:2]

[{'id': 'root::TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement-p-3',
  'text': '6 rue Adolphe Fischer L-1520 Luxembourg',
  'paragraph_enum': 0,
  'relationsCount': 0},
 {'id': 'root::TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement-p-4',
  'text': 'Luxembourg',
  'paragraph_enum': 1,
  'relationsCount': 0}]

In [13]:
def is_empty_value(value):
    return value in (None, "", [], {})


def merge_node_records(existing, new):
    merged = existing.copy()

    for key, value in new.items():
        if key not in merged or is_empty_value(merged[key]):
            merged[key] = value
        elif is_empty_value(value):
            continue
        elif merged[key] == value:
            continue
        else:
            conflicts = merged.setdefault("_merge_conflicts", {})
            conflicts.setdefault(key, [])
            if value not in conflicts[key]:
                conflicts[key].append(value)

    return merged


def merge_nodes(node_batches):
    merged_by_id = {}

    for batch in node_batches:
        for node in batch:
            node_id = node["id"]

            if node_id not in merged_by_id:
                merged_by_id[node_id] = node.copy()
            else:
                merged_by_id[node_id] = merge_node_records(merged_by_id[node_id], node)

    return sorted(merged_by_id.values(), key=lambda node: node["id"])


def merge_edges(edge_batches):
    merged = {}

    for batch in edge_batches:
        for edge in batch:
            edge_key = (edge["src"], edge["type"], edge["tgt"])
            if edge_key not in merged:
                merged[edge_key] = edge.copy()

    return sorted(merged.values(), key=lambda edge: (edge["src"], edge["type"], edge["tgt"]))


def merge_graph_results(results):
    valid_results = [result for result in results if result]

    contract_ids = []
    source_paragraph_ids = []
    node_batches = []
    edge_batches = []

    for result in valid_results:
        contract_id = result.get("contract_id")
        source_paragraph_id = result.get("source_paragraph_id")

        if contract_id and contract_id not in contract_ids:
            contract_ids.append(contract_id)
        if source_paragraph_id and source_paragraph_id not in source_paragraph_ids:
            source_paragraph_ids.append(source_paragraph_id)

        node_batches.append(result.get("nodes", []))
        edge_batches.append(result.get("edges", []))

    return {
        "contract_ids": contract_ids,
        "source_paragraph_ids": source_paragraph_ids,
        "nodes": merge_nodes(node_batches),
        "edges": merge_edges(edge_batches),
    }


# example_messages = format_extraction_messages(paragraph_inputs[0])
# example_messages["user"][:1500]

In [14]:
# example_messages

In [16]:
from dotenv import load_dotenv
import os
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
client = OpenAI(api_key=api_key)


def extract_graph_from_paragraph(paragraph_node, model="gpt-4.1-mini"):
    messages = format_extraction_messages(paragraph_node)

    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "system",
                "content": messages["system"],
            },
            {
                "role": "user",
                "content": messages["user"],
            },
        ],
    )

    return json.loads(response.output_text)


sample_result = extract_graph_from_paragraph(paragraph_inputs[0])
sample_result


{'contract_id': 'TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement',
 'source_paragraph_id': 'root::TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement-p-3',
 'nodes': [{'id': 'clause:root::TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement-p-3',
   'node_type': 'CLAUSE',
   'source_paragraph_id': 'root::TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement-p-3',
   'text': '6 rue Adolphe Fischer L-1520 Luxembourg',
   'paragraph_enum': 0,
   'clause_id': None,
   'title': None,
   'level': None},
  {'id': 'party:6 rue Adolphe Fischer L-1520 Luxembourg',
   'node_type': 'PARTY',
   'name': '6 rue Adolphe Fischer L-1520 Luxembourg',
   'role': None,
   'address': None}],
 'edges': [{'src': 'clause:root::TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding Agreement-p-3',
   'tgt': 'party:6 rue Adolphe Fischer L-1520 Luxembourg',
   'type': 'MENTIONS_PARTY'},
  {'src': 'clause:root::TomOnlineInc_200

In [ ]:
def extract_graph_batch(paragraph_inputs, model="gpt-5.5", limit=None):
    selected_inputs = paragraph_inputs if limit is None else paragraph_inputs[123:123 + limit]

    results = []
    errors = []

    for paragraph_node in tqdm(selected_inputs, desc="Extracting KG", unit="paragraph"):
        try:
            result = extract_graph_from_paragraph(paragraph_node, model=model)
            results.append(result)
        except Exception as exc:
            errors.append(
                {
                    "source_paragraph_id": paragraph_node["id"],
                    "error": str(exc),
                }
            )

    return {
        "results": results,
        "errors": errors,
        "merged_graph": merge_graph_results(results),
    }


batch_output = extract_graph_batch(paragraph_inputs, limit=3)
len(batch_output["results"]), len(batch_output["errors"]), len(batch_output["merged_graph"]["nodes"]), len(batch_output["merged_graph"]["edges"])


Extracting KG: 100%|██████████| 3/3 [02:16<00:00, 45.58s/paragraph]


(3, 0, 28, 55)

In [ ]:
def save_merged_graph(merged_graph, source_path, suffix="_knowledge_graph"):
    source_file = Path(source_path)
    output_path = source_file.with_name(f"{source_file.stem}{suffix}.json")

    output_payload = {
        "source_file": source_file.name,
        "contract_ids": merged_graph.get("contract_ids", []),
        "source_paragraph_ids": merged_graph.get("source_paragraph_ids", []),
        "nodes": merged_graph.get("nodes", []),
        "edges": merged_graph.get("edges", []),
    }

    output_path.write_text(
        json.dumps(output_payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    return output_path


merged_graph = batch_output["merged_graph"]
saved_graph_path = save_merged_graph(merged_graph, graph_test[4])
saved_graph_path


PosixPath('../../infra/json/graph/TomOnlineInc_20060501_20-F_EX-4.46_749700_EX-4.46_Co-Branding_Agreement_knowledge_graph.json')